In [1]:
from pymargo.core import Engine
import pyyokan_common as yokan
from pyyokan_client import Client
from pyyokan_server import Provider

import json
import ctypes
import struct
import blosc2
import numpy as np
import numpy as np
from pathlib import Path
import sys

In [2]:
sys.path.append('/projects/insituperf/seer_o/venv-seer/lib/python3.11/site-packages')

In [3]:
sys.path.append('/projects/insituperf/SZ3/tools/pysz/')
from pysz import SZ

In [4]:
import pyvista as pv

In [5]:
def list_all_keys(db):
    num_keys = db.count()
    
    max_length = 1024
    prefix = ''
    out_keys = []
    for i in range(0, num_keys):
      out_keys.append( bytearray(max_length+len(prefix)+1) )
    
    from_key = ''
    ksizes = db.list_keys(keys=out_keys, from_key=from_key, filter=prefix)
    
    keys = []
    for i in range(len(ksizes)):
        key_size = ksizes[i]
        
        k = out_keys[i]
        key = (k[:key_size]).decode('ascii')
        keys.append(key)
        
    return keys

In [6]:
def list_fields(db, simid):
    
    all_keys = list_all_keys(db)
    
    x = []
    for k in all_keys:
        parts = k.split('/')
        if parts[0] == '_' + str(simid):
            if len(parts) == 5:
                name = parts[len(parts)-2]
                x.append(name)
        
    return list(set(x))

In [7]:
def list_attributes(db, simid):
    all_keys = list_all_keys(db)

    x = []
    for k in all_keys:
        parts = k.split('/')
        if parts[0] == '_' + str(simid):
            if len(parts) == 5:
                name = parts[len(parts)-1]
                x.append(name)

    return list(set(x))

In [8]:
def get_value(db, key):
    ''' Get data from the server for that key '''

    # length of the value associated with the key
    l = db.length(key)

    out_val = bytearray(l)          # create buffer
    db.get(key=key, value=out_val)  # get the data
    v = out_val.decode("ascii")     # convert to ascii
    return v

In [9]:
def get_data(db, key):
    ''' Get data from the server for that key '''

    # length of the value associated with the key
    l = db.length(key)

    out_val = bytearray(l)          # create buffer
    db.get(key=key, value=out_val)  # get the data
    return out_val

In [10]:
def get_decompDataBLOSC(db, key, num_elems):
        x = []
        
        val = get_data(db, key)
        a_bytesobj2 = blosc2.decompress(val)
        
        bf = str(num_elems) + 'f'
        x = struct.unpack(bf, a_bytesobj2)

        return x

In [11]:
def get_decompDataSZ3(db, key, num_elems):
        data = get_data(db, key)
        np_array = np.frombuffer(data, dtype=np.uint8)
        
        lib_extention = {
            "darwin": "libSZ3c.dylib",
            "windows": "SZ3c.dll",
        }.get(sys.platform, "libSZ3c.so")

        sz = SZ("/projects/insituperf/SZ3/install/lib64/{}".format(lib_extention))
        
        data_dec = sz.decompress(np_array, (num_elems,1,1), np.float32)


        return data_dec

In [12]:
def isTsReady(db, ts, simid):
    key = '_' + simid + '/' + ts +'/status'
    return get_value(db, key)

In [13]:
def getNumRanks(db, simid):
    key = '_' + simid + '/num_ranks'
    return get_value(db, key)

In [16]:
f = open('/projects/insituperf/seer_o/test_app/mochi-yokan-config.json')
#f = open('/vast/projects/exasky/pascal/HACC/trunk/mochi-yokan-config.json')

In [17]:
json_data = json.load(f)
json_data

{'sim-id': '07720',
 'libraries': {'yokan': '/vast/home/pascalgrosset/spack/opt/spack/linux-rhel8-haswell/gcc-9.4.0/mochi-yokan-0.4.2-hiu7yh7om6nmyc2ahuknpdsov5k64zcj/lib/libyokan-bedrock-module.so'},
 'providers': [{'name': 'yokan_provider',
   'provider_id': 124,
   'type': 'yokan',
   'pool': '__primary__',
   'config': {'database': {'type': 'map'}}}],
 'data': [{'name': 'pressure_3', 'compressor': 'BLOSC'},
  {'name': 'temperature_3', 'compressor': 'SZ3', 'mode': 'psnr', 'value': 50}],
 'databases': [{'address': '192.168.81.77:44399',
   'protocol': 'ofi+tcp',
   'provider_id': 124},
  {'address': '192.168.81.78:45745',
   'protocol': 'ofi+tcp',
   'provider_id': 124}]}

In [18]:
server_addr1 = "ofi+tcp://192.168.81.77:44399"
provider_id = 124
protocol = 'ofi+tcp'

In [19]:
engine1 = Engine(protocol)
mid1 = engine1.get_internal_mid()
addr1 = engine1.lookup(server_addr1)
hg_addr1 = addr1.get_internal_hg_addr()
provider1 = Provider(mid=mid1, provider_id=provider_id, config='{"database":{"type":"map"}}')
client1 = Client(mid=mid1)
db1 = client1.make_database_handle(address=hg_addr1, provider_id=provider_id)

In [20]:
server_addr2 = "ofi+tcp://192.168.81.78:45745"
provider_id = 124
protocol = 'ofi+tcp'

In [21]:
engine2 = Engine(protocol)
mid2 = engine2.get_internal_mid()
addr2 = engine2.lookup(server_addr2)
hg_addr2 = addr2.get_internal_hg_addr()
provider2 = Provider(mid=mid2, provider_id=provider_id, config='{"database":{"type":"map"}}')
client2 = Client(mid=mid2)
db2 = client2.make_database_handle(address=hg_addr2, provider_id=provider_id)

In [22]:
dbs = []

In [23]:
dbs.append(db1)
dbs.append(db2)

In [24]:
ts = '1'

In [25]:
simid = '07720'

In [26]:
keys = list_all_keys(dbs[0])
keys

['_07718/0/0/pressure_3/compressed_size',
 '_07718/0/0/pressure_3/dbIndex',
 '_07718/0/0/pressure_3/num_elems',
 '_07718/0/0/pressure_3/type',
 '_07718/0/0/pressure_3/value',
 '_07718/0/0/temperature_3/compressed_size',
 '_07718/0/0/temperature_3/dbIndex',
 '_07718/0/0/temperature_3/num_elems',
 '_07718/0/0/temperature_3/type',
 '_07718/0/0/temperature_3/value',
 '_07718/0/1/pressure_3/compressed_size',
 '_07718/0/1/pressure_3/dbIndex',
 '_07718/0/1/pressure_3/num_elems',
 '_07718/0/1/pressure_3/type',
 '_07718/0/1/pressure_3/value',
 '_07718/0/1/temperature_3/compressed_size',
 '_07718/0/1/temperature_3/dbIndex',
 '_07718/0/1/temperature_3/num_elems',
 '_07718/0/1/temperature_3/type',
 '_07718/0/1/temperature_3/value',
 '_07718/0/2/pressure_3/compressed_size',
 '_07718/0/2/pressure_3/dbIndex',
 '_07718/0/2/pressure_3/num_elems',
 '_07718/0/2/pressure_3/type',
 '_07718/0/2/pressure_3/value',
 '_07718/0/2/temperature_3/compressed_size',
 '_07718/0/2/temperature_3/dbIndex',
 '_07718/0/2/

In [27]:
fields = list_fields(dbs[0], simid)
fields

['pressure_3', 'temperature_3']

In [28]:
attributes = list_attributes(dbs[0], simid)
attributes

['type', 'compressed_size', 'value', 'dbIndex', 'num_elems']

In [29]:
isTsReady(dbs[0], '1', simid)

'1'

In [30]:
getNumRanks(dbs[0], simid)

'4'

In [35]:
get_value(dbs[0], "_07720/0/1/pressure_3/dbIndex")

'0'

In [36]:
get_value(dbs[0], "_07720/1/1/pressure_3/dbIndex")

'1'

In [ ]:
nE = get_value(dbs[0],"_07720/0/1/pressure_3/num_elems")
nE

In [ ]:
get_value(dbs[0],"_07720/4/3/pressure_3/num_elems")

In [37]:
pressure_3 = get_decompDataBLOSC(dbs[0], "_07720/0/1/pressure_3/value", 100)
pressure_3

(50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0,
 50000.0)

In [38]:
pressure_3 = get_decompDataBLOSC(dbs[1], "_07720/1/1/pressure_3/value", 100)
pressure_3

(50000.0,
 50010.0,
 50020.0,
 50030.0,
 50040.0,
 50050.0,
 50060.0,
 50070.0,
 50080.0,
 50090.0,
 50100.0,
 50110.0,
 50120.0,
 50130.0,
 50140.0,
 50150.0,
 50160.0,
 50170.0,
 50180.0,
 50190.0,
 50200.0,
 50210.0,
 50220.0,
 50230.0,
 50240.0,
 50250.0,
 50260.0,
 50270.0,
 50280.0,
 50290.0,
 50300.0,
 50310.0,
 50320.0,
 50330.0,
 50340.0,
 50350.0,
 50360.0,
 50370.0,
 50380.0,
 50390.0,
 50400.0,
 50410.0,
 50420.0,
 50430.0,
 50440.0,
 50450.0,
 50460.0,
 50470.0,
 50480.0,
 50490.0,
 50500.0,
 50510.0,
 50520.0,
 50530.0,
 50540.0,
 50550.0,
 50560.0,
 50570.0,
 50580.0,
 50590.0,
 50600.0,
 50610.0,
 50620.0,
 50630.0,
 50640.0,
 50650.0,
 50660.0,
 50670.0,
 50680.0,
 50690.0,
 50700.0,
 50710.0,
 50720.0,
 50730.0,
 50740.0,
 50750.0,
 50760.0,
 50770.0,
 50780.0,
 50790.0,
 50800.0,
 50810.0,
 50820.0,
 50830.0,
 50840.0,
 50850.0,
 50860.0,
 50870.0,
 50880.0,
 50890.0,
 50900.0,
 50910.0,
 50920.0,
 50930.0,
 50940.0,
 50950.0,
 50960.0,
 50970.0,
 50980.0,
 50990.0)

In [ ]:
len(com_x)

In [ ]:
val_x = get_decompDataSZ3(dbs[0], "_07115/499/0/x/value", 21344230)

In [ ]:
vals_x = val_x.flatten()

In [ ]:
vals_x

In [ ]:
val_y = get_decompDataSZ3(dbs[0], "_07115/499/0/y/value", 21344230)
vals_y = val_y.flatten()

In [ ]:
val_z = get_decompDataSZ3(dbs[0], "_07115/499/0/z/value", 21344230)
vals_z = val_z.flatten()

In [ ]:
xxx= np.stack([vals_x,vals_y,vals_z], axis=1)

In [ ]:
xxx

In [ ]:
np.savetxt("/projects/insituperf/seer_o/3d_array.csv", xxx, delimiter=",")

In [ ]:
np.save('/projects/insituperf/seer_o/my_array.npy', xxx)

In [ ]:
import pyvista as pv

In [ ]:
%pip list

In [ ]:
!{sys.executable} -m pip install 'pyvista[jupyter]>=0.38.1'